# Local Transformer Inference for Token Classification

Today we will peform token classification using a locally executed transformer encoder.

UNlike statistical sequence models such as Hidden Markov Models, transformer encoders produce contextual neural \
representations for every token in a sequence. 

We will implement:
- Local transformer model loading

- Hidden state extraction

- Subword-to-word alignment

- Final word-level tag reconstruction

### Transformer Token Classification Pipeline

The worflow follows:

Input Senetence -> Tokenizer -> Subword Tokens -> Transformer Encoder -> Last Hidden State Tensor ->

Classification Head -> Subword Predictions -> Word ALignment -> Final Word-Level Tags

### Requirements

1. Local Transformer Execution

    Load a lightweight pretrained token classification model locally using Hugging Face Transformers.

    Extract:
    - Tokenized inputs
    
    - Last hidden state tensor
    
    - Token classification logits

    No high-level inference pipelines may be used.

2. Manual Classification

    For each subword token:
    - Extract logits
    
    - Apply argmax manually
    
    - Convert class IDs into tag labels

    This exposes the complete token classification process.

3. Subword Alignment

    Transformer tokeinizers frequently split words into mutiple fragments.

    Example:

    embeddings -> embed   ##dings

    Subword predictions must be aligned back to the original word sequence using tokenizer word mappings.

4. Alignment Resolution Strategy

    When multiple subwords belong to the same original word:
    - Use the first subword prediction
    
    - Ignore subsequent continuation fragments

    This produces a single tag per original word.

### Expected Output

```text
LOCAL TRANSFORMER INFERENCE (v1)

MODEL CONFIGURATION

Base Encoder:
dslim/bert-base-NER

Hidden Dimension (d_model):
768

Subword Token Count:
...

SUBWORD ALIGNMENT ANALYSIS

Token: the
Word ID: 0
Tag: O

Token: dense
Word ID: 1
Tag: O


FINAL ALIGNED OUTPUT

the ↔ O
dense ↔ O
matrix ↔ O
captured ↔ O
semantic ↔ O
embeddings ↔ B-MISC
```

### Imports

In [55]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForTokenClassification

### Validation Input

In [56]:
sentence = "the dense matrix captured semantic embeddings"
model_name = "dslim/bert-base-NER"

### Tokenizer and Model Loader

In [57]:
def load_model_components(model_name):
    """Loads tokenizer and token classification model"""

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = (AutoModelForTokenClassification.from_pretrained(model_name))

    # Select GPU if available, otherwise use CPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    return tokenizer, model, device

### Tokenization Engine

In [58]:
def tokenize_sentence(tokenizer, sentence):
    """Tokenizes text while preserving word-to-subword alignment metadata"""

    encoded = tokenizer(
        sentence.split(),
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True
    )

    return encoded

### Hidden State Extraction Engine

In [59]:
def extract_hidden_states(encoded, model):
    """Runs transformer inference and extract hidden state representations"""

    with torch.no_grad():
        outputs = model(**encoded, output_hidden_states=True)
    
    hidden_states = (outputs.hidden_states[-1])
    logits = outputs.logits

    return hidden_states, logits

### Token Classification Engine

In [60]:
def classify_subwords(logits, model):
    """Converts logits into predicted token classification labels"""

    predicted_ids = torch.argmax(logits, dim=-1)[0]
    labels = []

    for idx in predicted_ids:
        labels.append(model.config.id2label[idx.item()])

    return labels

### Subwords Alignment Engine

In [61]:
def align_predictions(encoded, tokenizer, predicted_labels, original_sentence):
    """Aligns subword predictions back to original whitespace-separated words"""

    word_ids = encoded.word_ids()
    tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
    aligned = {}
    diagnostics = []

    for token, word_id, label in zip(tokens, word_ids, predicted_labels):
        if word_id is None:
            continue

        diagnostics.append((token, word_id, label))

        # First-subword strategy
        if word_id not in aligned:
            aligned[word_id] = label

    words = original_sentence.split()
    final_output = []

    for idx, word in enumerate(words):
        final_output.append((word, aligned[idx]))

    return diagnostics, final_output

### Evaluation Harness

In [62]:
def evaluate_local_transformer_pipeline(sentence, model_name):
    """Executes the complete local transformer token classification pipeline"""

    print("LOCAL TRANSFORMER INFERENCE (v1)\n")

    tokenizer, model, device = (load_model_components(model_name))
    encoded = tokenize_sentence(tokenizer, sentence)
    encoded = encoded.to(device)
    hidden_states, logits = extract_hidden_states(encoded, model)
    predicted_labels = classify_subwords(logits, model)
    diagnostics, final_output = (align_predictions(encoded, tokenizer, predicted_labels, sentence))

    print("MODEL CONFIGURATION\n")
    print(f"Base encoder: {model_name}")
    print(f"Device: {device}")
    print(f"Hidden Dimension (d_model): {hidden_states.shape[-1]}")
    print(f"Subword Token Count: {hidden_states.shape[1]}\n")

    print("SUBWORD ALIGNMENT ANALYSIS\n")

    for token, word_id, label in diagnostics:
        print(
            f"Token: {token:<12} | Word ID: {word_id} | Tag: {label}"
        )

    print("\nFINAL ALIGNED OUTPUT\n")

    for word, label in final_output:
        print(f"{word} <-> {label}")

### Execute Local Transformer Pipeline

In [65]:
evaluate_local_transformer_pipeline(sentence, model_name)

LOCAL TRANSFORMER INFERENCE (v1)



Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


MODEL CONFIGURATION

Base encoder: dslim/bert-base-NER
Device: cpu
Hidden Dimension (d_model): 768
Subword Token Count: 11

SUBWORD ALIGNMENT ANALYSIS

Token: the          | Word ID: 0 | Tag: O
Token: dense        | Word ID: 1 | Tag: O
Token: matrix       | Word ID: 2 | Tag: O
Token: captured     | Word ID: 3 | Tag: O
Token: semantic     | Word ID: 4 | Tag: O
Token: em           | Word ID: 5 | Tag: O
Token: ##bed        | Word ID: 5 | Tag: O
Token: ##ding       | Word ID: 5 | Tag: O
Token: ##s          | Word ID: 5 | Tag: O

FINAL ALIGNED OUTPUT

the <-> O
dense <-> O
matrix <-> O
captured <-> O
semantic <-> O
embeddings <-> O
